# Spoke bronze layer: Incremental Auto Loader ingestion of "rides" json files

## Defining variables

In [0]:
import uuid
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())

SOURCE = "/Volumes/spoke/bronze/landing/rides/"
SCHEMA_LOCATION = "/Volumes/spoke/ops/checkpoints/rides/schema"
CHECKPOINT = "/Volumes/spoke/ops/checkpoints/rides/checkpoint"
TARGET = "spoke.bronze.rides"

## Defining schema

In [0]:
SCHEMA = ("""
          bike_id STRING, 
          device STRUCT<app_version: STRING, platform: STRING>, 
          duration_seconds STRING, 
          end_station_id STRING, 
          ended_at STRING, 
          issues ARRAY<STRING>, 
          ride_id STRING, 
          start_station_id STRING, 
          started_at STRING, 
          user_type STRING,
          _corrupt_record STRING
          """)

## Reading json files

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .schema(SCHEMA)
      .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
      .option("cloudFiles.schemaEvolutionMode", "rescue")
      .option("rescuedDataColumn", "_rescued_data")
      .option("columnNameOfCorruptRecord", "_corrupt_record")
      .load(SOURCE)
)

## Adding metadata columns

In [0]:
df = (df
      .withColumn("_ingested_at", F.current_timestamp())
      .withColumn("_source_file", F.col("_metadata.file_path"))
      .withColumn("_ingest_batch_id", F.lit(RUN_ID))
      .withColumn("_source_file_modification_time", F.col("_metadata.file_modification_time"))
)

## Write to Delta table

In [0]:
(df.writeStream
 .format("delta")
 .option("checkpointLocation", CHECKPOINT)
 .trigger(availableNow=True)
 .toTable(TARGET)
)